# LLM Pass: Basic Emotions Extraction

Batch-extracts basic-emotion coding (Anger, Contempt, Disgust, Fear,
Happiness, Sadness, Surprise) from crisis comments via Gemma 4 31B
(Gemini API). Same batched-JSON machinery as the grief-outrage pass.

Conventions: resume-by-id checkpointing, split-and-retry to zero skipped
rows, raw JSONL archive, `run_log.json` provenance per run.

In [ ]:
crisis_name = 'ateneo-drowning'
model = 'gemma-4-31b-it'
input_dir = '../../01-preprocessing/output/ateneo-drowning/youtube_backup'
sources = ['04-institutional_response-uaap_ban']
output_dir = '../output/_experiments/basic-emotions'
raw_dir = '../output/_experiments/raw/basic-emotions'
batch_size = 25
workers = 2
rpm_target = 12
max_retries = 3
max_comment_chars = None
temperature = 0.0
seed = 42
max_output_tokens = 4096
force = False
sample_n = 2000
sample_seed = 42


In [ ]:
import hashlib
import json
import os
import threading
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from google import genai
from google.genai import types

CWD = Path.cwd()
INPUT_DIR = (CWD / input_dir).resolve()
OUTPUT_DIR = (CWD / output_dir).resolve()
RAW_DIR = (CWD / raw_dir).resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RAW_DIR.mkdir(parents=True, exist_ok=True)

load_dotenv(CWD / '..' / '..' / '.env')
api_key = os.environ.get('GEMINI_API_KEY', '')
if not api_key:
    raise RuntimeError('GEMINI_API_KEY missing from environment')
client = genai.Client(api_key=api_key)
print('SDK:', genai.__version__)

SYSTEM_INSTRUCTION = (
    'I will provide a dataset of comments in CSV format containing an `id` '
    'and `comment` column. You must return the analysis strictly as a JSON '
    'array of objects. Do not include markdown formatting (other than the '
    'JSON block) or explanations outside of the JSON.\n\n'
    'For each comment, extract:\n'
    '(a) the emotion(s) expressed, classified strictly using only one or '
    'more of these 7 categories: Anger, Contempt, Disgust, Fear, '
    'Happiness, Sadness, Surprise.\n'
    '(b) the value or relationship invoked as the grounds for the emotion.\n'
    "(c) whose family is referenced (e.g., the victims' family, the "
    "commenter's own family, or any parent).\n\n"
    'Use this exact JSON structure for each object:\n'
    '- `id`: The ID from the CSV.\n'
    '- `basic_emotions`: An array of strings containing only the allowed '
    '7 emotions. If no emotion is clear, output an empty array.\n'
    '- `value_or_relationship_invoked`: A string explaining the grounds. '
    'If none, state "None".\n'
    '- `family_referenced`: An array of strings specifying the family. '
    'If none, return an empty array.'
)

RESPONSE_SCHEMA = {
    'type': 'array',
    'items': {
        'type': 'object',
        'properties': {
            'id': {'type': 'string'},
            'basic_emotions': {'type': 'array', 'items': {'type': 'string'}},
            'value_or_relationship_invoked': {'type': 'string'},
            'family_referenced': {'type': 'array', 'items': {'type': 'string'}},
        },
        'required': ['id', 'basic_emotions',
                       'value_or_relationship_invoked', 'family_referenced'],
    },
}

OUTPUT_COLUMNS = ['id', 'basic_emotions',
                    'value_or_relationship_invoked', 'family_referenced']
ARRAY_COLUMNS = {'basic_emotions', 'family_referenced'}
ALLOWED_EMOTIONS = {'Anger', 'Contempt', 'Disgust', 'Fear',
                      'Happiness', 'Sadness', 'Surprise'}


def build_config():
    # Full structured-output config; degrade gracefully on older SDKs.
    try:
        return types.GenerateContentConfig(
            system_instruction=SYSTEM_INSTRUCTION,
            temperature=temperature,
            seed=seed,
            response_mime_type='application/json',
            response_schema=RESPONSE_SCHEMA,
            thinking_config=types.ThinkingConfig(thinking_level='MINIMAL'),
            max_output_tokens=max_output_tokens,
        )
    except Exception as e:
        print(f'Full config unsupported ({e}); temperature-only fallback.')
        return types.GenerateContentConfig(
            system_instruction=SYSTEM_INSTRUCTION,
            temperature=temperature,
            seed=seed,
        )


CONFIG = build_config()
PROMPT_SHA = hashlib.sha256(SYSTEM_INSTRUCTION.encode()).hexdigest()[:12]
print(f'Model: {model} | prompt_sha: {PROMPT_SHA}')
print(f'Input: {INPUT_DIR} | Output: {OUTPUT_DIR}')


In [ ]:
def is_degenerate(text):
    # Empty or punctuation-only rows can never match the schema usefully.
    return not isinstance(text, str) or not any(ch.isalnum() for ch in text)


def empty_record(row_id):
    return {'id': row_id, 'basic_emotions': [],
            'value_or_relationship_invoked': 'None', 'family_referenced': []}


def load_source(stem):
    df = pd.read_csv(INPUT_DIR / f'{stem}.csv')
    df = df.reset_index(drop=True)
    if sample_n is not None and len(df) > sample_n:
        df = df.sample(n=sample_n, random_state=sample_seed).reset_index(drop=True)
    df['id'] = [f'{stem}-{i:05d}' for i in range(len(df))]
    df['comment'] = df['text'].fillna('')
    degenerate = df['comment'].apply(is_degenerate)
    defaults = {r['id']: empty_record(r['id'])
                for _, r in df[degenerate].iterrows()}
    work = df[~degenerate].copy()
    truncated = {}
    if max_comment_chars:
        # Long-tail cost control; flagged for sensitivity analysis.
        over = work['comment'].str.len() > max_comment_chars
        truncated = dict(zip(work.loc[over, 'id'], work.loc[over, 'comment']))
        work.loc[over, 'comment'] = work.loc[over, 'comment'].str[:max_comment_chars]
    # Dedup identical texts; aliases rejoin without extra API calls.
    work = work.drop_duplicates(subset='comment').reset_index(drop=True)
    alias = {}
    for _, r in df[~degenerate].iterrows():
        first = work.loc[work['comment'] == r['comment'], 'id']
        if len(first) and first.iloc[0] != r['id']:
            alias.setdefault(first.iloc[0], []).append(r['id'])
    print(f'{stem}: rows={len(df)} degenerate={int(degenerate.sum())} '
          f'unique={len(work)} truncated={len(truncated)}')
    return work[['id', 'comment']], defaults, alias, truncated


SOURCES = {}
if isinstance(sources, str):
    sources = [sources]
for stem in sources:
    SOURCES[stem] = load_source(stem)


In [ ]:
_pacer_lock = threading.Lock()
_last_call = [0.0]
MIN_INTERVAL = 60.0 / rpm_target


def pace():
    # Shared token-bucket pacing keeps workers under RPM without hot-loop sleep.
    with _pacer_lock:
        wait = MIN_INTERVAL - (time.monotonic() - _last_call[0])
        if wait > 0:
            time.sleep(wait)
        _last_call[0] = time.monotonic()


def is_safety_block(response):
    try:
        cands = getattr(response, 'candidates', None) or []
        fr = str(getattr(cands[0], 'finish_reason', '')) if cands else ''
        fb = str(getattr(response, 'prompt_feedback', ''))
        blob = (fr + fb).upper()
        return any(k in blob for k in ('SAFETY', 'BLOCK', 'RECITATION',
                                        'PROHIBITED'))
    except Exception:
        return False


def parse_records(text, expected_ids):
    body = text.strip()
    if body.startswith('```'):
        body = body.split('\n', 1)[1] if '\n' in body else body[3:]
        body = body.rsplit('```', 1)[0].strip()
    records = json.loads(body)
    if not isinstance(records, list):
        raise ValueError('response is not a JSON array')
    got = [r.get('id') for r in records]
    if got != expected_ids:
        raise ValueError(f'id mismatch: want {len(expected_ids)}, got {len(got)}')
    for r in records:
        bad = set(r.get('basic_emotions', [])) - ALLOWED_EMOTIONS
        if bad:
            raise ValueError(f'emotion outside allowed 7: {bad} in {r}')
    return records


def call_chunk(chunk_df):
    # One billed call; returns (records|None, usage, error|None).
    payload = chunk_df[['id', 'comment']].to_csv(index=False)
    expected = chunk_df['id'].tolist()
    usage = {'prompt_tokens': 0, 'completion_tokens': 0, 'calls': 0}
    for attempt in range(max_retries):
        try:
            pace()
            t0 = time.monotonic()
            response = client.models.generate_content(
                model=model,
                contents='Data:\n' + payload,
                config=CONFIG,
            )
            usage['calls'] += 1
            usage['seconds'] = usage.get('seconds', 0) + time.monotonic() - t0
            meta = getattr(response, 'usage_metadata', None)
            if meta is not None:
                usage['prompt_tokens'] += getattr(meta, 'prompt_token_count', 0) or 0
                usage['completion_tokens'] += getattr(
                    meta, 'candidates_token_count', 0) or 0
            if is_safety_block(response):
                return None, usage, 'safety-block: fail fast, no retry'
            text = response.text
            if not text or not text.strip():
                raise ValueError('empty response text')
            return parse_records(text, expected), usage, None
        except Exception as e:
            msg = str(e).upper()
            if '429' in msg or 'RESOURCE_EXHAUSTED' in msg:
                time.sleep(90 if 'TOKEN' in msg else 60)
                continue
            if any(k in msg for k in ('500', '503', 'INTERNAL', 'UNAVAILABLE',
                                       'TIMEOUT', 'RESET', 'CONNECTION')):
                time.sleep(10)
                continue
            if attempt < max_retries - 1:
                time.sleep(15.0 * (attempt + 1))
    return None, usage, f'exhausted {max_retries} attempts'


def resolve_batch(batch_df, depth=0):
    # Recursive split-and-retry: one bad row cannot poison the whole batch.
    records, usage, err = call_chunk(batch_df)
    if records is not None:
        return records, [usage], None
    if len(batch_df) == 1 or depth >= 3:
        return None, [usage], err
    mid = len(batch_df) // 2
    out, usages = [], []
    for half in (batch_df.iloc[:mid], batch_df.iloc[mid:]):
        r, u, e = resolve_batch(half, depth + 1)
        usages += u
        if r is None:
            return None, usages, e
        out += r
    return out, usages, None


In [ ]:
print('Sanity check: tiny 5-row prober before the billable loop')
stem0 = sources[0]
work0, _, _, _ = SOURCES[stem0]
sanity_df = work0.head(5)
recs, usages, err = resolve_batch(sanity_df)
assert recs is not None, f'sanity batch failed: {err}'
assert len(recs) == len(sanity_df), 'sanity coverage mismatch'
print(f'OK: {len(recs)} records, calls={sum(u["calls"] for u in usages)}')
print(json.dumps(recs[:2], indent=1, ensure_ascii=False)[:1200])


In [ ]:
def out_csv(stem):
    return OUTPUT_DIR / f'llm_basic-emotions_{stem}.csv'


def records_to_df(by_id):
    rows = []
    for rid in sorted(by_id):
        r = dict(by_id[rid])
        for col in ARRAY_COLUMNS:
            if not isinstance(r.get(col), str):
                r[col] = json.dumps(r.get(col, []), ensure_ascii=False)
        rows.append({c: r.get(c, '') for c in OUTPUT_COLUMNS})
    return pd.DataFrame(rows, columns=OUTPUT_COLUMNS)


def load_done(stem):
    # Resume map: id -> record, arrays parsed back from JSON strings.
    path = out_csv(stem)
    if force or not path.exists():
        return {}
    prev = pd.read_csv(path, keep_default_na=False)
    by_id = {}
    for _, row in prev.iterrows():
        rec = dict(row)
        for col in ARRAY_COLUMNS:
            try:
                rec[col] = json.loads(rec[col]) if rec[col] else []
            except Exception:
                rec[col] = []
        by_id[rec['id']] = rec
    return by_id


RUN_STATS = {}
for stem in sources:
    work, defaults, alias, truncated = SOURCES[stem]
    by_id = load_done(stem)
    by_id.update(defaults)
    pending = work[~work['id'].isin(by_id)].reset_index(drop=True)
    batches = [pending.iloc[i:i + batch_size]
               for i in range(0, len(pending), batch_size)]
    print(f'{stem}: pending={len(pending)} in {len(batches)} batches '
          f'(resumed {len(by_id) - len(defaults)} rows)')
    token_use = {'prompt_tokens': 0, 'completion_tokens': 0, 'calls': 0,
                 'seconds': 0.0}
    failed, succeeded = [], 0
    raw_path = RAW_DIR / f'{stem}.jsonl'
    with ThreadPoolExecutor(max_workers=workers) as pool:
        future_of = {pool.submit(resolve_batch, b): b for b in batches}
        done = 0
        for fut in as_completed(future_of):
            batch = future_of[fut]
            recs, usages, err = fut.result()
            for u in usages:
                for k in token_use:
                    token_use[k] += u.get(k, 0)
            if recs is None:
                failed.append((batch, err))
            else:
                succeeded += len(recs)
                for r in recs:
                    by_id[r['id']] = r
                with open(raw_path, 'a', encoding='utf-8') as fh:
                    for r in recs:
                        fh.write(json.dumps(r, ensure_ascii=False) + '\n')
            done += 1
            records_to_df(by_id).to_csv(out_csv(stem), index=False)
            print(f'  [{done}/{len(batches)}] ok_rows={succeeded} '
                  f'failed_batches={len(failed)}')
    # Sequential retry pass: every failed batch until zero skipped or logged.
    still_failed = []
    for batch, err in failed:
        recs, usages, err2 = resolve_batch(batch)
        for u in usages:
            for k in token_use:
                token_use[k] += u.get(k, 0)
        if recs is None:
            still_failed.append((batch['id'].tolist(), err2))
        else:
            for r in recs:
                by_id[r['id']] = r
    records_to_df(by_id).to_csv(out_csv(stem), index=False)
    # Expand dedup aliases so every original id has a row.
    final_df = records_to_df(by_id)
    extra = []
    for first_id, dup_ids in alias.items():
        if first_id in by_id:
            for dup in dup_ids:
                row = dict(by_id[first_id])
                row['id'] = dup
                extra.append(row)
    if extra:
        final_df = pd.concat([final_df, records_to_df({r['id']: r for r in extra})],
                             ignore_index=True)
    final_df.to_csv(out_csv(stem), index=False)
    RUN_STATS[stem] = {'rows': len(final_df), 'failed': still_failed,
                       'tokens': token_use, 'truncated': len(truncated)}
    print(f'{stem}: DONE rows={len(final_df)} '
          f'unresolved={len(still_failed)} '
          f"tokens={token_use['prompt_tokens']}+{token_use['completion_tokens']}")


In [ ]:
from itertools import combinations
from collections import Counter

for stem in sources:
    df = pd.read_csv(out_csv(stem), keep_default_na=False)
    emo_lists = df['basic_emotions'].apply(
        lambda s: json.loads(s) if s else [])
    print(f'== {stem} (n={len(df)}) ==')
    print('label histogram:')
    print(pd.Series([e for lst in emo_lists for e in lst])
          .value_counts().to_string())
    multi = (emo_lists.str.len() > 1).mean()
    print(f'\nmulti-label rate: {multi:.2f} '
          f'(mean {emo_lists.str.len().mean():.2f} emotions/comment)')
    pairs = Counter()
    for lst in emo_lists:
        for a, b in combinations(sorted(set(lst)), 2):
            pairs[(a, b)] += 1
    print('\ntop co-occurring pairs:')
    for (a, b), c in pairs.most_common(10):
        print(f'  {a} + {b}: {c}')
    print('\ntop grounds:')
    print(df['value_or_relationship_invoked'].value_counts().head(10).to_string())
    print()


In [ ]:
run_log = {
    'pass': 'basic-emotions',
    'model': model,
    'temperature': temperature,
    'seed': seed,
    'prompt_sha256': PROMPT_SHA,
    'sdk': genai.__version__,
    'batch_size': batch_size,
    'sample': {'n': sample_n, 'seed': sample_seed},
    'crisis': crisis_name,
    'sources': sources,
    'ran_at': datetime.now(timezone.utc).isoformat(),
    'stats': RUN_STATS,
}
with open(OUTPUT_DIR / 'run_log.json', 'w', encoding='utf-8') as fh:
    json.dump(run_log, fh, indent=1, default=str)
print('run_log.json written')
for stem in sources:
    print(f'{stem}: {out_csv(stem)}')
